# 4D Coil Sketching: Example code to test different regularizations

Install environment first to use using the instructions in README.md. If you uncover a potential bug installing cudnn (this has happened to me before), then you can run the following in the command line to manually install (be sure to activate your correct environment first):

`python -m cupyx.tools.install_library --library cudnn --cuda 12.x`

This will install cudnn for cuda version 12.x.

WARNING: This is the pre-release version. The clean version is in progress.

# Setup and data loading


In [ ]:
import os
import sys
sys.path.append('../src')
sys.path.append('../sigpy_mod')  # Add the subdirectory to Python path
import numpy as np
import matplotlib.pyplot as plt
import sigpy      as sp
import sigpy.mri  as mr
import sigpy.plot as pl
from utils import coil_compression, crop_center, crop_2d, create_montage
import nibabel as nib
from monitor_gpu import monitor_gpu_memory  # GPU monitoring

# Setup
devnum = 0 # Device to use, set to -1 if running in CPU
toeplitz=False # Use Toeplitz to speed up A.N evaluations (i.e. the normal linear operator)
compress = True # Use SVD coil compression to compress and reorder data according to coil energy contribution
combine_csm = True # If you want to calculate 1 CSM from all combined data # Recommended to reduce memory, especially if using Toeplitz
espirit_csm = True # Use ESPIRIT instead of JSENSE (0.55T MRI data seemed to perform more reliably with ESPIRIT, but JSENSE is still a good option)
# Note: if espirit_csm is False, recon will use JSENSE. JSENSE requires installation of cudnn. This can be achieved using the terminal or manually in Python. Contact joseph.plummer@nih.gov for help.

# Number of sketched coils (including the constant non-sketched coils)
nch = 3 
number_non_sketched_coils = nch-1 

# Additional parameters
R = 1 # Throw away some data if attempting a faster reconstruction R>=1.
lamda = 0.5 # Recommended 0.001 to 1.
scale = 1 # Intensity scale the images by this factor.
res_scale = 1 # Resolution scale, e.g. 0.5 means 50% of the resolution in each dimension.
coil_percentile = 100 # Keep only the top X% of coils by energy, e.g. 100 means all coils are kept, 50 means only the top 50% of coils are kept.

# List of recons
nufft=True
cg_sense = True
pdhg_sense = True
gm_sense = True
wavelet = True
tv = True
lor = True
mocolor = True
llor = True

# Device configuration
try:
    device = sp.Device(devnum)
except:
    print("Original devnum failed, using devnum 0 instead.")
    devnum=0
    device = sp.Device(devnum)
xp     = device.xp
device.use()
mvd = lambda x: sp.to_device(x, devnum)
mvc = lambda x: sp.to_device(x, sp.cpu_device)

# Load data
folder = 'data/'
additional_spatial_encoding = 32 # Optional additional spatial encoding in the x-y dimensions to help reduce wrapping artifacts (very useful if the shoulders/neck extend beyond the reconstructed FOV)
matrix_e =  (288+(additional_spatial_encoding),288+(additional_spatial_encoding),128)
print(f'Reconstructing onto a matrix size of: {matrix_e}. This includes an additional {additional_spatial_encoding} voxels in the x and y dimensions to help reduce wrapping artifacts.')
matrix_r = tuple(int(dim * res_scale) for dim in matrix_e) # Reconstructed image size, may be different from matrix_e if res_scale != 1
matrix_c = tuple(int(dim * res_scale) for dim in (288, 288, 112))  # Cropped image size
matrix_a = tuple(int(dim) for dim in (288, 288, 128))  # For density compensation
matrix_i = (288, 288)
ksp = np.load(folder + "bksp.npy")
coord = np.load(folder + "bcoord.npy")
coord[...,0] = coord[...,0]*matrix_e[0]
coord[...,1] = coord[...,1]*matrix_e[1]
coord[...,2] = coord[...,2]*matrix_e[2]
[nphase, nc0, nviews0, nread] = ksp.shape
print(f'Phases: {nphase}; Coils: {nc0}; Excitations: {nviews0}; Samples: {nread}')    

# Undersample spirals
nviews = int(nviews0/R)
ksp = ksp[...,:nviews,:]
coord = coord[:,:nviews,...] 

# Make a new folder to store the lamda value
if not os.path.exists(folder + f"/lamda_{lamda:.2e}"):
    os.makedirs(folder + f"/lamda_{lamda:.2e}/")
folder = folder + f"/lamda_{lamda:.2e}/"

# Undersample sample points along readout if res_scale < 1
nf_arr = np.sqrt(np.sum(coord[0, 0, :, :]**2, axis=1)) 
nf_e = np.sum(nf_arr < np.max(nf_arr)*res_scale)
print(f"Subsetting first {100*res_scale:.1f}% nf_e") # Note, this is not robust for VDS where the edges are more sampled than center
ksp = ksp[...,:nf_e]
coord = coord[...,:nf_e,:]
[nphase, nc, nviews, nread] = ksp.shape
print(f'SUBSAMPLED: Phases: {nphase}; Coils: {nc}; Excitations: {nviews}; Samples: {nread}') 

# Check whether a specified save data path exists
results_exist = os.path.exists(folder + "/sketching/png")

# Create a new directory because the results path does not exist
if not results_exist:
    os.makedirs(folder + "/sketching/png")
    print("A new directory inside: " + folder +
            " called 'sketching/png' has been created.")

# Save images as Nifti files
# Custom affine for coronally acquired images
aff = np.array([[0, 1, 0, 0],
                [0, 0, 1, 0],
                [1, 0, 0, 0],
                [0, 0, 0, 1]])
print("Affine transformation matrix used for saving this data: ")
print(str(aff))

# View coords
plt.figure(figsize=(3, 3), dpi=100)
ax = plt.axes(projection='3d')
N_visual = nviews//200  # Number of projections you want to show, inefficient for large n
color = iter(plt.cm.viridis(np.linspace(0, 1, N_visual)))
for i in np.linspace(0, nviews-1, N_visual):
    i = int(i)
    c = next(color)
    ax.scatter(coord[0, i, :, 0], coord[0, i, :, 1],
               coord[0, i, :, 2], color=c, s=0.5, marker='.')
ax.set_zlabel('$k_z$')
ax.set_ylabel('$k_y$')
ax.set_xlabel('$k_x$')
plt.axis("off")
# plt.savefig("coord.png", transparent=True, bbox_inches='tight', pad_inches=0)
plt.show()

def normalize(img, percentile=90):
    img_n = img/np.percentile(img.ravel(), percentile)
    return img_n

## [OPTIONAL]: apply a Hanning filter to raw k-space data to reduce high frequency noise 

For highly noisy images, this could be useful in enhancing edges. You can speed it up by moving to the GPU using `cupy` functionality.



In [ ]:
from scipy.signal.windows import tukey
ksp_filter_3d = False

def apply_tukey_filter(ksp, coords, tukey_alpha=0.2):
    """
    Applies a Tukey filter to k-space data based on the spatial location in kx, ky, kz.

    Parameters:
        ksp (np.ndarray): K-space data of shape (nc, nexc, nread).
        coords (np.ndarray): Coordinates of k-space samples of shape (nexc, nread, 3).
        tukey_alpha (float): Alpha parameter for the Tukey filter (0 = rectangular, 1 = Hann).

    Returns:
        np.ndarray: Filtered k-space data of the same shape as input ksp.
    """
    # Initialize filtered k-space array
    filtered_ksp = np.copy(ksp)

    # Find the max range in each dimension for normalization
    max_kx, max_ky, max_kz = np.max(np.abs(coords[..., 0])), np.max(np.abs(coords[..., 1])), np.max(np.abs(coords[..., 2]))
    
    # Create Tukey windows in each k-space dimension
    tukey_kx = tukey(len(np.unique(coords[..., 0])), tukey_alpha)
    tukey_ky = tukey(len(np.unique(coords[..., 1])), tukey_alpha)
    tukey_kz = tukey(len(np.unique(coords[..., 2])), tukey_alpha)
    
    # Normalize coordinates to index ranges in each dimension
    for i in range(coords.shape[0]):
        for j in range(coords.shape[1]):
            # Normalize kx, ky, kz to indices in Tukey windows
            idx_kx = int(((coords[i, j, 0] / max_kx) + 1) / 2 * (len(tukey_kx) - 1))
            idx_ky = int(((coords[i, j, 1] / max_ky) + 1) / 2 * (len(tukey_ky) - 1))
            idx_kz = int(((coords[i, j, 2] / max_kz) + 1) / 2 * (len(tukey_kz) - 1))
            
            # Apply the Tukey filter to the corresponding k-space point
            filter_value = tukey_kx[idx_kx] * tukey_ky[idx_ky] * tukey_kz[idx_kz]
            filtered_ksp[:, i, j] *= filter_value

    return filtered_ksp

if ksp_filter_3d:
    for i in range(nphase):
        ksp[i, ...] = apply_tukey_filter(ksp[i,...], coord[i, ...], tukey_alpha=0.1)
        print(f'3D Tukey filter applied to phase {i}.')


## Estimate k-space preconditioner

We opted for Frank Ong's k-space preconditioner versus the more traditional Pipe-Menon DCF, as it is more robust for iterative reconstructions with large numbers of iterations. Adjust accordingly.

In [ ]:
# Estimate k-space preconditioner
print("Calculating preconditioner...")
dens = np.zeros((nphase, 1, nviews, nread))
use_dcf = False # Use Pipe-Menon density compensation function instead of preconditioning (not as robust as preconditioning, but may offer prettier images)
t2star_weighting = False # Additional T2* weighting to the density compensation function
for resp in range(nphase):
    if use_dcf:
        dens[resp,0,...] = mvc(mr.pipe_menon_dcf(coord[resp,...], matrix_a, device=device, max_iter=30))
        
        # Multiply by a Tukey window to reduce high frequency noise
        from scipy.signal.windows import tukey
        tukey_window = tukey(2*nread, alpha=0.1)  # Alpha is the shape parameter for Tukey
        dens_weighting = tukey_window[nread:]
        dens[resp,0,...] *= dens_weighting

    else:
        # Precondition using Frank Ong's convex optimization problem 
        mps_precond = np.ones((1,) + matrix_a) # TODO: decide on the correct matrix size to use here
        mps_precond /= len(mps_precond)**0.5
        dens[resp,0,...] = mvc(mr.kspace_precond(
                            mps_precond,
                            coord=sp.to_device(coord[resp,...], device),
                            device=sp.Device(device), lamda=1e-2, oversamp=1.5))
        
    if t2star_weighting:
        # Estimate T2* decay	
        t2_star = 9  # ms # Assumed for 0.55T MRI data, adjust as needed	
        readout = 5 # ms	
        dwell_time = readout/nread	
        relaxation = np.zeros((nread,))	
        for jj in range(nread):	
            relaxation[jj] = np.exp(-(jj*dwell_time)/t2_star)**0.5
        dens[resp,0,:,:] *= relaxation
print(f"dens.shape: {dens.shape}")

# Normalize all to the same relative intensity
dens /= np.median(dens) 

# Plot
fig, (ax1) = plt.subplots(
    1, figsize=(4, 4), dpi=100)
for resp in range(nphase):
    ax1.plot(np.arange(nread), dens[resp, 0, 0,:], label=f'phase: {resp}') 
ax1.set_ylabel('dens')
if use_dcf:
    ax1.set_title("Pipe-Menon density compensation")
else:
    ax1.set_title("single channel k-space preconditioner")
ax1.set_xlabel("sample number")
plt.legend()
plt.show()

## Perform SVD coil compression

This works by first performing SVD on the raw k-space data, and then by reordering the data according to the size of each Eigenvalue. This allows you to "subset" the X number of coils with the largest contribution. However, for coil sketching, we keep all data without subsetting. We just perform the SVD part to reorder k-space data from high to low coil energy. The motivation for this comes from the coil sketching "selection" process, where higher energy coils are preferentially selected over lower energy coils. This allows faster convergence as the reconstruction favors coils with larger contribution. There are setups where this may not be favorable. For instance, if you tuned your coil selection process to be entirely random (Uniformly distributed). 

In [ ]:
# Coil compression steps
if compress:
    if combine_csm: # WARNING: may not work as well if lots of excitations shared
        nc0 = ksp.shape[1]
        print("Running coil compression on all data combined into one array...")
        ksp_full = np.zeros((nc0,nphase*nviews,nread), dtype=complex)
        for coil in range(nc0): # Run through each coil individually
            # TODO: remove duplicate rows
            ksp_full[coil, ...] = ksp[:,coil,...].reshape((nphase*nviews,nread))
        ksp_full = mvd(ksp_full)
        print(f"ksp.shape before coil compression: {ksp.shape}")
        ksp_full, eigenvalues, nc = coil_compression(ksp_full, percentile=coil_percentile)
        ksp_full = mvc(ksp_full)
        for coil in range(nc0): # Run through each coil individually
            ksp[:,coil, ...] = ksp_full[coil,...].reshape((nphase, nviews,nread))
        print(f"eigenvalues[...,-4,-3,-2,-1] following coil-compression: ...{eigenvalues[-4:]}")
        ksp = mvc(ksp)
        del ksp_full
        del coil_compression
    else:       
        print("Running coil compression on each respiratory frame...")
        ksp = mvd(ksp[:,:,:nviews,...])
        print(f"ksp.shape before coil compression: {ksp.shape}")
        for resp in range(nphase):
            ksp[resp,...], eigenvalues, nc = coil_compression(ksp[resp,...], percentile=coil_percentile)
            print(f"phase {resp}: eigenvalues[...,-4,-3,-2,-1] following coil-compression: ...{eigenvalues[-4:]}")
        ksp = mvc(ksp)
        
if compress:
    if nc0 == 15:
        print(f"LARGE COIL COMPRESSING TO {nc} COILS")
        nc = nc
    elif nc0 == 21:
        nc = nc # 18
        print(f"MEDIUM COIL COMPRESSING TO {nc} COILS")
                
    if 'pdw' in folder:
        nc = nc0
        print("NOT REDUCING COILS AS PDW IMAGE AND AIMING TO MAXIMIZE SIGNAL")


# Subsample coil channels and declare variables
ksp_us = ksp[:,:nc,:nviews,...] 
coord_us = coord[:,:nviews,...] 
dens_us = dens[:,:,:nviews,...]
del(ksp)


## Estimate coil sensitivity maps

In this code block, we use J-SENSE NLINV (built in SigPy feature for non-Cartesian data) to estimate coil sensitivities for each respiratory phase individually (i.e. when `combine_csm = False`). For most cases on our end, we found that the coil sensitivity estimations were more in line with expectations when we first combined all respiratory phase data together, and then estimated the maps on the larger k-space array. This may vary site-to-site, application to application.

In [ ]:
print("Calculating csm...")
mps = np.zeros([nphase] + [nc] + list(matrix_r), dtype=complex)
if combine_csm == False:
    for resp in range(nphase):
        mps[resp,...] = mvc(sp.mri.app.JsenseRecon(ksp_us[resp,:nc,...], 
                                                coord=coord_us[resp,...],
                                                weights=dens_us[resp,0,...], # TODO: decide if helpful
                                                mps_ker_width=5, # Smaller leads to lower res mps, but maybe good if smoothing over lungs
                                                ksp_calib_width=20, 
                                                lamda=1e-2,
                                                img_shape=matrix_r, 
                                                device=device, 
                                                max_iter=30, 
                                                max_inner_iter=20, 
                                                show_pbar=True).run())
        # mps[resp,...] = mps_precond

    sp.plot.ImagePlot(mps[resp], x=1, y=2, z=0,
                                title=f"[MAG] Sensitivity maps estimated with J-SENSE NLINV for respiratory phase {resp}", colormap='gray', mode='m')
    sp.plot.ImagePlot(mps[resp], x=1, y=2, z=0,
                                title=f"[PHASE] Sensitivity maps estimated with J-SENSE NLINV for respiratory phase {resp}", colormap='hsv', mode='p')


    ni_img = nib.Nifti1Image(crop_center(abs(np.moveaxis(mps[0,...], 0, -1)), matrix_c), affine=aff)
    nib.save(ni_img, folder + '/sketching/csm_phase_0')

    import cupy as cp
    cp._default_memory_pool.free_all_blocks()

## Attempt to calculate CSM using JSENSE (Cartesian) or ESPIRIT

In this code block, we perform more experimentation with sensitivity map estimations. This time, we use a custom function that automatically grids the non-Cartesian data onto a Cartesian grid, and then uses J-SENSE or ESPIRIT reconstruction to estimate coil maps. Depending on application, this approach could be favorable. It is worth exploring in more detail.

In [ ]:
sys.path.append('../sigpy_mod')
import csm
import importlib
importlib.reload(csm)
if combine_csm is False:
    if espirit_csm:

        print("Calculating csm...")
        mps_jsense = np.zeros([nphase] + [nc] + list(matrix_r), dtype=complex)
        mps_espirit = np.zeros([nphase] + [nc] + list(matrix_r), dtype=complex)
        for resp in range(nphase):
            mps_jsense[resp,...] = csm.jsense_csm(ksp_us[resp,:nc,...], coord_us[resp,...], dens_us[resp,0,...], matrix_r, device,
                                            mps_ker_width=5,
                                            ksp_calib_width=24,
                                            lamda=1e-4,)
            mps_espirit[resp,...] = csm.espirit_csm(ksp_us[resp,:nc,...], coord_us[resp,...], dens_us[resp,0,...], matrix_r, device, 
                            crop=0,
                            thresh=0.02,
                                kernel_width=6,
                                calib_width=24,
                                max_iter=100).get()
            
        sp.plot.ImagePlot(mps_jsense[resp,...], x=1, y=2, z=0,
                                    title=f"MPS jsense respiratory phase {resp}", colormap='gray', mode='m')
        sp.plot.ImagePlot(mps_jsense[resp,...], x=1, y=2, z=0,
                                    title=f"MPS jsense respiratory phase {resp}", colormap='hsv', mode='p')
        sp.plot.ImagePlot(mps_espirit[resp,...], x=1, y=2, z=0,
                                    title=f"[MAG] MPS espirit respiratory phase {resp}", colormap='gray', mode='m')    
        sp.plot.ImagePlot(mps_espirit[resp,...], x=1, y=2, z=0,
                                    title=f"[PHASE] MPS espirit respiratory phase {resp}", colormap='hsv', mode='p')


        # FORCEFULLY OVERWRITE MPS WITH OUR NEW ONE
        # mps = mps_jsense
        mps = mps_espirit
        del mps_espirit, mps_jsense

        import cupy as cp
        cp._default_memory_pool.free_all_blocks()


        # rss = np.linalg.norm(mps[resp, :,...], axis=0) 

        # sp.plot.ImagePlot(rss, x=1, y=2, z=0,
        #                             title=f"rss", colormap='gray', mode='m')

## Estimate coil sensitivity maps from all respiratory-phase data combined into one array

Here, we estimate the coil sensitivity maps on all k-space data combined into one array. For our purposes, this worked most successfully, as it was less susceptible to undersampling artifacts caused by the relatively-under-sampled respiratory-phase k-space data. Again, it is worth examining for your own data to see which approach works best. Or, you could use your own sensitivity maps. Please share if you have a superior approach!

In [ ]:
if combine_csm:
    importlib.reload(csm)
    print("Calculating csm for all phases combined together (with dcf)...")
    ksp_full = np.zeros((nc,nphase*nviews,nread), dtype=complex)
    for coil in range(nc): # Run through each phase individually
        ksp_full[coil, ...] = ksp_us[:,coil,...].reshape((nphase*nviews,nread))
    coord_full = coord_us.reshape((nphase*nviews,nread, 3))
    
    if espirit_csm:
        dens_full = mvc(mr.pipe_menon_dcf(coord_full, matrix_a, device=device, max_iter=30, width=3.5, beta=8))
        tukey_window = tukey(2*nread, alpha=0.1)  # Alpha is the shape parameter for Tukey
        dens_weighting = tukey_window[nread:]
        dens_full *= dens_weighting
        mps_full = csm.espirit_csm(ksp_full, coord_full, dens_full, matrix_r, device, 
                        crop=0,
                        thresh=0.02,
                            kernel_width=6,
                            calib_width=24-4,
                            max_iter=100,
                            downscale_factor=2).get()
        sp.plot.ImagePlot(mps_full, x=1, y=2, z=0,
                                title=f"[MAG] Sensitivity maps estimated with ESPIRIT for ALL phases", colormap='gray', mode='m',vmax=1)
        sp.plot.ImagePlot(mps_full, x=1, y=2, z=0,
                                    title=f"[PHASE] Sensitivity maps estimated with ESPIRIT for ALL phases", colormap='hsv', mode='p')
        sp.plot.ImagePlot(mps_full-mps[resp], x=1, y=2, z=0,
                                    title=f"[MAG] Sensitivity maps estimated with ESPIRIT (DIFFERENCE)", colormap='gray', mode='m', vmax=0.2)
        
    else:
        mps_precond = np.ones((1,) + matrix_a) # TODO: decide on the correct matrix size to use here
        mps_precond /= len(mps_precond)**0.5
        dens_full = mvc(mr.kspace_precond(
                                    mps_precond,
                                    coord=sp.to_device(coord_full[...], device),
                                    device=sp.Device(device), lamda=1e-2, oversamp=1.5))[0,...]
        mps_full = mvc(sp.mri.app.JsenseRecon(ksp_full, 
                                            coord=coord_full, 
                                            weights=dens_full, 
                                            mps_ker_width=8-3, 
                                            ksp_calib_width=24-4, 
                                            lamda=1e-2,
                                            img_shape=matrix_r, 
                                            device=device, 
                                            max_iter=30,
                                            max_inner_iter=20, 
                                            show_pbar=True).run())
        sp.plot.ImagePlot(mps_full, x=1, y=2, z=0,
                                title=f"[MAG] Sensitivity maps estimated with J-SENSE NLINV for ALL phases", colormap='gray', mode='m',vmax=1)
        sp.plot.ImagePlot(mps_full, x=1, y=2, z=0,
                                    title=f"[PHASE] Sensitivity maps estimated with J-SENSE NLINV for ALL phases", colormap='hsv', mode='p')
        sp.plot.ImagePlot(mps_full-mps[resp], x=1, y=2, z=0,
                                    title=f"[MAG] Sensitivity maps estimated with J-SENSE NLINV (DIFFERENCE)", colormap='gray', mode='m', vmax=0.2)
    del(ksp_full, coord_full, dens_full)



    for resp in range(nphase):
        mps[resp,...] = mps_full
        print("Overwriting each respiratory phase's sensitivity map with the full map.")
        
    ni_img = nib.Nifti1Image(crop_center(abs(np.moveaxis(mps[0,...], 0, -1)), matrix_c), affine=aff)
    nib.save(ni_img, folder + '/sketching/csm_phase_combined')
        
import cupy as cp
cp._default_memory_pool.free_all_blocks()

## Let's do some linear algebra 

We can perform a simple "gridded" reconstruction of the raw k-space data. This is a good way to examine how under-sampled your data is, visually.

In [ ]:
print(f'ksp_us.shape: {ksp_us.shape}')
print(f'coord_us.shape: {coord_us.shape}')
print(f'dens_us.shape: {dens_us.shape}')
print(f'mps.shape: {mps.shape}')
print(ksp_us[0,...].shape[:-3])

# 4D NUFFT
F = sp.linop.NUFFT(ishape=(nc,) + matrix_r,
                coord=coord_us[0,...])
print("done")
S = sp.linop.Multiply(ishape = (nc,) + matrix_r, # Keep the channel as the ishape to ensure the output image is multiple channels
                    mult=mps[0,...])
print("done")
D = sp.linop.Multiply(F.oshape, dens_us[0,...])
print("done")
A_us = D * F * S
print("done")

y = ksp_us[0,...]
y = sp.to_device(y, 1)
img_us = mvc(A_us.H(y))
y = sp.to_device(y, sp.cpu_device)
del y
scale = np.percentile(np.abs(img_us.flatten()), 95)
vmax = np.percentile(np.abs(img_us.flatten()), 99)
print(r'shape of reconstructed image using $A^H y$' + f' is: {img_us.shape}')
pl.ImagePlot(img_us, x=1, y=2, z=0,vmax=vmax, title="Single respiratory frame inverse NUFFT", colormap="gray")
del(F,S,D,A_us)     

import cupy as cp
cp._default_memory_pool.free_all_blocks()

## Let's do a bunch of GPU cleanup 

(I'm not sure which of these lines is optimal, I just know that running all of them generally clears the GPU). Someone smarter at `cupy`, please help!

In [ ]:
import gc
gc.collect()

import torch
import importlib

# Get memory usage in MB
print(f"Allocated memory: {torch.cuda.memory_allocated() / 1024 ** 2} MB")
print(f"Cached memory: {torch.cuda.memory_reserved() / 1024 ** 2} MB")

import cupy as cp

# Get memory usage statistics
mempool = cp.get_default_memory_pool()
pinned_mempool = cp.get_default_pinned_memory_pool()

print(f"Used memory: {mempool.used_bytes() / 1024 ** 2} MB")
print(f"Total memory: {mempool.total_bytes() / 1024 ** 2} MB")

# To free unused memory
mempool.free_all_blocks()

# Optionally clear any pinned memory (CPU-GPU pinned transfers)
pinned_mempool.free_all_blocks()

# Clear variables stored on GPU 
cp._default_memory_pool.free_all_blocks()

# Avoiding Device Synchronization for Faster Execution
# By default, many CuPy operations perform implicit device synchronization (waiting for all 
# previous operations to complete before continuing). You can explicitly manage synchronization 
# to improve performance by deferring synchronization calls:
cp.cuda.stream.get_current_stream().synchronize()  # Manually synchronize


print(f"Used memory: {mempool.used_bytes() / 1024 ** 2} MB")
print(f"Total memory: {mempool.total_bytes() / 1024 ** 2} MB")


In [ ]:
import importlib

# Reload cupy to see if that is where the memory dump is located
importlib.reload(cp)
importlib.reload(sp)
importlib.reload(mr)
import time
time.sleep(10) # Wait a second to give it time to clear memory if not done already 

print(f"Used memory: {mempool.used_bytes() / 1024 ** 2} MB")
print(f"Total memory: {mempool.total_bytes() / 1024 ** 2} MB")



## Let's try a sketched recon using our 4D $A$ operator

In [ ]:
import sketching_4d_app as sk4d
import importlib
importlib.reload(sk4d)
importlib.reload(sp)

# Coil sketching reconstruction
max_init_iter = 4 # Initialize the recon with a few iterations using only the nch highest energy coils (similar to a coil compressed reconstruction).
max_inner_iter = 5 # Number of inner iterations per coil subset, for the sketched data consistency term. Can be reduced for very good preconditioning, and increased for poor preconditioning.
max_outer_iter = 10 # Overall number of outer iterations. If you have more sketched coils, you may be able to decrease this number. This generally worked well for nch=3-5. The higher you make it, the more random coil subsets will be included in the iterative reconstruction. In theory, a very high number will support near identical image reconstruction to a fully sampled conventional reconstruction.
np.random.seed(1)

# Clear variables stored on GPU 
import cupy as cp
cp._default_memory_pool.free_all_blocks()


## Combine sensitivity maps into one (if chosen)

If using only a single coil sensitivity map for all respiratory phases (recommended for our test data), then collapse `mps` into a 4D array.


In [ ]:
if combine_csm and len(mps.shape) == 5:
    mps = mvc(mps)[0,...]
else:
    mps = mvc(mps)

## Choose whether to save objective values

Set to true if you would like to estimate the value of the overall objective function at each iteration (note, this does not use coil sketching to evaluate). This is useful if plotting convergence curves. 

WARNING: This is slow and memory intensive!!

In [ ]:
save_objective_values=False

## CG-SENSE coil sketched reconstruction

A custom implementation of CG-SENSE. While fast, this likely offers little benefit compared to a gridded reconstruction, as it is not using the temporal dimension to regularize.


In [ ]:
if cg_sense:
    # Move variables to cpu to free up space
    ksp_us = mvc(ksp_us)
    coord_us = mvc(coord_us)
    dens_us = mvc(dens_us)

        
    import sketching_4d_app as sk4d
    import sketching_app
    importlib.reload(sketching_app)
    import importlib
    importlib.reload(sk4d)
    importlib.reload(sp)
    sys.path.append('../sigpy_mod')
    import custom_linop
    import importlib
    importlib.reload(custom_linop)
    import maxeig as me
    importlib.reload(me)
    
    # Start monitoring
    print("MONITORING GPU USAGE...")
    tic, toc, memory_usage_sketched, shutdown = monitor_gpu_memory(gpu_index=devnum)
    tic()
    
    if save_objective_values:
        ksp_us = mvd(ksp_us)
        dens_us = mvd(dens_us)
    sketched_cg_solver = sk4d.SketchedSenseRecon(ksp_us/scale, 
                                                mps=mps, 
                                                lamda=lamda*1e3, # TODO: get MaxEig normalization to work robustly for CG-SENSE, enabling a more robust choice of lamda. For now, this works.
                                                reduced_ncoils=nch,
                                    weights=dens_us, 
                                    coord=coord_us,
                                    solver='ConjugateGradient',
                                    toeplitz=toeplitz,
                                    show_pbar=True, device=sp.Device(devnum), # Must be same device as ksp and dcf 
                                    number_non_sketched_coils=number_non_sketched_coils, # None = Defaults to nch-1 
                                    max_init_iter=max_init_iter,
                                    max_outer_iter=max_outer_iter,
                                    max_inner_iter=max_inner_iter,
                                    max_power_iter=30, # Not used for conjugate gradient 
                                    save_objective_values=save_objective_values, # This is memory intensive as it requires saving y on the backend, and computing A^Hy at every iteration
                                    coil_batch_size=nch,
                                    )
    
    img_sk_cg_sense = mvc(scale*sketched_cg_solver.run())
    if save_objective_values:
        sketched_cg_sense_objective = np.array(sketched_cg_solver.objective_values)
    del sketched_cg_solver
    
    sketched_memory_cg = np.array(memory_usage_sketched)*1e-3
    toc()
    # Plot results
    plt.figure(figsize=(3,4))
    plt.plot(memory_usage_sketched)
    plt.xlabel('time')
    plt.ylabel('memory usage (MB)')
    plt.grid(True)
    plt.show()
    shutdown()
    print("GPU MONITORING COMPLETE.")
        
    ni_img = nib.Nifti1Image(crop_center(abs(np.moveaxis(img_sk_cg_sense, 0, -1)), matrix_c), affine=aff)
    nib.save(ni_img, folder + '/sketching/img_sk_cg_sense')

    # Clear variables stored on GPU 
    import cupy as cp
    cp._default_memory_pool.free_all_blocks()

    import sigpy.plot as pl
    vmax = np.percentile(abs(img_sk_cg_sense.ravel()), 95)
    pl.ImagePlot(img_sk_cg_sense[0,...], 
                x=1, y=3,
                title="Under-sampled Sketching Reconstruction (CG SENSE)", 
                colormap="gray", 
                vmax=vmax,
                )
    
    pl.ImagePlot(img_sk_cg_sense[0,...], 
                x=1, y=3, mode='p',
                title="PHASE Under-sampled Sketching Reconstruction (CG SENSE)", 
                colormap="jet", 
                )
    
    # Get memory usage statistics
    mempool = cp.get_default_memory_pool()
    pinned_mempool = cp.get_default_pinned_memory_pool()
    mempool.free_all_blocks()
    pinned_mempool.free_all_blocks()
    cp._default_memory_pool.free_all_blocks()
    cp.cuda.stream.get_current_stream().synchronize() 
    
    # Display sagittal slices from insp/exp phases
    print(f"Sketched reconstruction with {nch} coils:")
    fig, ax = plt.subplots(1, 1, figsize=(3, 3))  # wider to accommodate two images side-by-side
    sag_slice = 6* matrix_c[0] // 10
    img_tmp = crop_center(np.moveaxis(img_sk_cg_sense, 0, -1),(matrix_c[0], matrix_c[1], int(res_scale*matrix_e[2])))
    img_insp = np.rot90(normalize(abs(img_tmp[..., nphase//2])), k=1)  
    img_exp = np.rot90(normalize(abs(img_tmp[..., 0])), k=1)
    sag_exp = np.rot90(img_exp[sag_slice, :, :], k=2)
    sag_insp = np.rot90(img_insp[sag_slice, :, :], k=2)
    concat_img = np.concatenate((sag_exp, sag_insp), axis=1)
    im = ax.imshow(concat_img, cmap='gray', vmax=1)
    ax.axis('off')
    fig.savefig(folder + f'/sketching/png/sketched_img_{nch}_coils_cg_sense.png', bbox_inches='tight', pad_inches=0)
    plt.show()
    

    
    # Move variables to cpu to free up space
    ksp_us = mvc(ksp_us)
    coord_us = mvc(coord_us)
    dens_us = mvc(dens_us)
    


## $L_1$ wavelet transform regularized reconstruction using Gradient Method

Perform a spatial L1-wavelet reconstruction, using Gradient Method as the solver. Note, this is also solvable with PDHG or FISTA.

Additionally, an extra reconstruction is shown for the "conventional" reconstruction (i.e. no coil sketching). This is akin to only running the initial iterations of the overall coil sketching algorithm, but using k-space data from all coils (instead of a subset of coils, as would be done for coil compression).

In [ ]:
if wavelet:
    # Move variables to cpu to free up space
    ksp_us = mvc(ksp_us)
    coord_us = mvc(coord_us)
    dens_us = mvc(dens_us)
        
    import sketching_4d_app as sk4d
    import importlib
    importlib.reload(sk4d)
    importlib.reload(sp)
    sys.path.append('../sigpy_mod')
    import custom_linop
    import importlib
    importlib.reload(custom_linop)
    img_sk_wavelet = mvc(scale*sk4d.SketchedL1WaveletRecon4D(ksp_us/scale, 
                                                mps=mps, 
                                                lamda=lamda, 
                                                reduced_ncoils=nch,
                                    weights=dens_us, 
                                    coord=coord_us,
                                    solver='GradientMethod',
                                    toeplitz=toeplitz,
                                    show_pbar=True, device=sp.Device(devnum), # Must be same device as ksp and dcf 
                                    number_non_sketched_coils=number_non_sketched_coils, # None = Defaults to nch-1 
                                    max_init_iter=max_init_iter,
                                    max_outer_iter=max_outer_iter,
                                    max_inner_iter=max_inner_iter,
                                    max_power_iter=30,
                                    coil_batch_size=nch).run())
    # img_sk = np.rot90(mvc(img_sk), k=-1)

    ni_img = nib.Nifti1Image(crop_center(abs(np.moveaxis(img_sk_wavelet, 0, -1)), matrix_c), affine=aff)
    nib.save(ni_img, folder + '/sketching/img_sk_wavelet')

    # Clear variables stored on GPU 
    import cupy as cp
    cp._default_memory_pool.free_all_blocks()
    import sigpy.plot as pl
    vmax = np.percentile(abs(img_sk_wavelet.ravel()), 95)
    pl.ImagePlot(img_sk_wavelet[3,...], 
                x=1, y=3,
                title="Under-sampled Sketching Reconstruction (Wavelet)", 
                colormap="gray", 
                vmax=vmax,
                )
    
    # Display sagittal slices from insp/exp phases
    print(f"Sketched reconstruction with {nch} coils:")
    fig, ax = plt.subplots(1, 1, figsize=(3, 3))  # wider to accommodate two images side-by-side
    sag_slice = 6* matrix_c[0] // 10
    img_tmp = crop_center(np.moveaxis(img_sk_wavelet, 0, -1),(matrix_c[0], matrix_c[1], int(res_scale*matrix_e[2])))
    img_insp = np.rot90(normalize(abs(img_tmp[..., nphase//2])), k=1)  
    img_exp = np.rot90(normalize(abs(img_tmp[..., 0])), k=1)
    sag_exp = np.rot90(img_exp[sag_slice, :, :], k=2)
    sag_insp = np.rot90(img_insp[sag_slice, :, :], k=2)
    concat_img = np.concatenate((sag_exp, sag_insp), axis=1)
    im = ax.imshow(concat_img, cmap='gray', vmax=1)
    ax.axis('off')
    fig.savefig(folder + f'/sketching/png/sketched_img_{nch}_coils_wavelet.png', bbox_inches='tight', pad_inches=0)
    plt.show()
    
    img_wavelet = mvc(scale*sk4d.SketchedL1WaveletRecon4D(ksp_us/scale, 
                                                mps=mps, 
                                                lamda=lamda, 
                                                reduced_ncoils=nc,
                                    weights=dens_us, 
                                    coord=coord_us,
                                    solver='GradientMethod',
                                    toeplitz=toeplitz,
                                    show_pbar=True, device=sp.Device(devnum), # Must be same device as ksp and dcf 
                                    number_non_sketched_coils=nc-1, # None = Defaults to nch-1 
                                    max_init_iter=max_init_iter + (max_outer_iter*(max_inner_iter-1)),
                                    max_outer_iter=0,
                                    max_inner_iter=0,
                                    max_power_iter=30,
                                    coil_batch_size=nc).run())
    # img_sk = np.rot90(mvc(img_sk), k=-1)

    ni_img = nib.Nifti1Image(crop_center(abs(np.moveaxis(img_wavelet, 0, -1)), matrix_c), affine=aff)
    nib.save(ni_img, folder + '/sketching/img_wavelet')

    # Clear variables stored on GPU 
    import cupy as cp
    cp._default_memory_pool.free_all_blocks()
    import sigpy.plot as pl
    vmax = np.percentile(abs(img_wavelet.ravel()), 95)
    pl.ImagePlot(img_wavelet[3,...], 
                x=1, y=3,
                title="Under-sampled Reconstruction (Wavelet)", 
                colormap="gray", 
                vmax=vmax,
                )


## Total variation reconstruction (using PDHG)
Supposedly, this is faster than the gradient method used in the L1 wavelet recon.

Supposedly, it is also more robust to non-smooth regularizations such as the nuclear norm.

Conveniently, I was able leverage $G$ for the motion fields, and $proxg()$ for the nuclear norm proximal operator used in MoCo-LR (later).

In [ ]:
if tv:
    import sketching_4d_app as sk4d
    import importlib
    importlib.reload(sk4d)
    importlib.reload(sp)

    # Change device to free up memory
    img_sk_tv = mvc(scale*sk4d.SketchedTotalVariationRecon4D(y=ksp_us/scale, 
                                                mps=mps, 
                                                lamda=lamda, 
                                                reduced_ncoils=nch,
                                    weights=dens_us, 
                                    coord=coord_us,
                                    solver='PrimalDualHybridGradient', # Default
                                    toeplitz=toeplitz,
                                    show_pbar=True, device=sp.Device(devnum),
                                    number_non_sketched_coils=number_non_sketched_coils, # None = Defaults to nch-1 
                                    max_init_iter=max_init_iter,
                                    max_outer_iter=max_outer_iter,
                                    max_inner_iter=max_inner_iter,
                                    max_power_iter=30,
                                    # TODO: Add max_cg_iter to control the conj grad iteration count. Default 10
                                    save_objective_values=False, # This is memory intensive as it requires saving y on the backend, and computing A^Hy at every iteration
                                    coil_batch_size=nch,
                                    ).run())

    ni_img = nib.Nifti1Image(crop_center(abs(np.moveaxis(img_sk_tv, 0, -1)), matrix_c), affine=aff)
    nib.save(ni_img, folder + '/sketching/img_sk_tv')
    # Clear variables stored on GPU 
    import cupy as cp
    cp._default_memory_pool.free_all_blocks()
    import sigpy.plot as pl
    vmax = np.percentile(abs(img_sk_tv.ravel()), 95)
    pl.ImagePlot(img_sk_tv[3,...], 
                x=1, y=3,
                title="Under-sampled Sketching Reconstruction (TV)", 
                colormap="gray", 
                vmax=vmax
                )
    
    # Display sagittal slices from insp/exp phases
    print(f"Sketched reconstruction with {nch} coils:")
    fig, ax = plt.subplots(1, 1, figsize=(3, 3))  # wider to accommodate two images side-by-side
    sag_slice = 6* matrix_c[0] // 10
    img_tmp = crop_center(np.moveaxis(img_sk_tv, 0, -1),(matrix_c[0], matrix_c[1], int(res_scale*matrix_e[2])))
    img_insp = np.rot90(normalize(abs(img_tmp[..., nphase//2])), k=1)  
    img_exp = np.rot90(normalize(abs(img_tmp[..., 0])), k=1)
    sag_exp = np.rot90(img_exp[sag_slice, :, :], k=2)
    sag_insp = np.rot90(img_insp[sag_slice, :, :], k=2)
    concat_img = np.concatenate((sag_exp, sag_insp), axis=1)
    im = ax.imshow(concat_img, cmap='gray', vmax=1)
    ax.axis('off')
    fig.savefig(folder + f'/sketching/png/sketched_img_{nch}_coils_tv.png', bbox_inches='tight', pad_inches=0)
    plt.show()
    
    img_tv = mvc(scale*sk4d.SketchedTotalVariationRecon4D(y=ksp_us/scale, 
                                                mps=mps, 
                                                lamda=lamda, 
                                                reduced_ncoils=nc,
                                    weights=dens_us, 
                                    coord=coord_us,
                                    solver='PrimalDualHybridGradient', # Default
                                    toeplitz=toeplitz,
                                    show_pbar=True, device=sp.Device(devnum),
                                    number_non_sketched_coils=nc-1, # None = Defaults to nch-1 
                                    max_init_iter=max_init_iter + ((max_inner_iter-1) * max_inner_iter),
                                    max_outer_iter=0,
                                    max_inner_iter=0,
                                    max_power_iter=30,
                                    # TODO: Add max_cg_iter to control the conj grad iteration count. Default 10
                                    save_objective_values=False, # This is memory intensive as it requires saving y on the backend, and computing A^Hy at every iteration
                                    coil_batch_size=nc,
                                    ).run())

    ni_img = nib.Nifti1Image(crop_center(abs(np.moveaxis(img_tv, 0, -1)), matrix_c), affine=aff)
    nib.save(ni_img, folder + '/sketching/img_tv')
    # Clear variables stored on GPU 
    import cupy as cp
    cp._default_memory_pool.free_all_blocks()
    import sigpy.plot as pl
    vmax = np.percentile(abs(img_tv.ravel()), 95)
    pl.ImagePlot(img_tv[3,...], 
                x=1, y=3,
                title="Under-sampled Reconstruction (TV)", 
                colormap="gray", 
                vmax=vmax
                )

## Let's try a 4D Low Rank recon (without registration)

Here, we try a custom implementation of a (globally) Low Rank reconstruction. I recommend using PDHG as the solver as it is more robust to under-sampling, however, Gradient Method also worked effectively for most cases in our site.

In [ ]:
if lor:
    import sketching_4d_app as sk4d
    import importlib
    importlib.reload(sk4d)
    importlib.reload(sp)
    import prox_lr
    importlib.reload(prox_lr)
    
    # Start monitoring
    print("MONITORING GPU USAGE...")
    tic, toc, memory_usage_sketched, shutdown = monitor_gpu_memory(gpu_index=devnum)
    tic()

    np.random.seed(1)
    img_sk_lr_no_moco = mvc(scale*sk4d.SketchedLowRankRecon4D(y=ksp_us/scale, 
                                                mps=mps, 
                                                lamda=lamda, 
                                                reduced_ncoils=nch,
                                    weights=dens_us, 
                                    coord=coord_us,
                                    moco=False,
                                    # solver='GradientMethod', # Default
                                    solver='PrimalDualHybridGradient', # Default
                                    toeplitz=toeplitz,
                                    show_pbar=True, device=sp.Device(devnum),
                                    number_non_sketched_coils=number_non_sketched_coils, # None = Defaults to nch-1 
                                    max_init_iter=max_init_iter,
                                    max_outer_iter=max_outer_iter,
                                    max_inner_iter=max_inner_iter,
                                    max_power_iter=30,
                                    save_objective_values=False, # This is memory intensive as it requires saving y on the backend, and computing A^Hy at every iteration
                                    coil_batch_size=nch,
                                    ).run())

    ni_img = nib.Nifti1Image(crop_center(abs(np.moveaxis(img_sk_lr_no_moco, 0, -1)), matrix_c), affine=aff)
    nib.save(ni_img, folder + '/sketching/img_sk_lr_no_moco')
    
    sketched_memory_lor = np.array(memory_usage_sketched)*1e-3
    toc()
    # Plot results
    plt.figure(figsize=(3,4))
    plt.plot(memory_usage_sketched)
    plt.xlabel('time')
    plt.ylabel('memory usage (MB)')
    plt.grid(True)
    plt.show()
    shutdown()
    print("GPU MONITORING COMPLETE.")

    # Clear variables stored on GPU 
    import cupy as cp
    cp._default_memory_pool.free_all_blocks()

    import sigpy.plot as pl
    vmax = np.percentile(abs(img_sk_lr_no_moco.ravel()), 95)
    for resp in range(nphase):
        pl.ImagePlot(img_sk_lr_no_moco[resp,...], 
                    x=1, y=3,
                    title="Under-sampled Sketching Reconstruction (LR - no registration)", 
                    colormap="gray", 
                    vmax=vmax,
                    )
        
    # Display sagittal slices from insp/exp phases
    print(f"Sketched reconstruction with {nch} coils:")
    fig, ax = plt.subplots(1, 1, figsize=(3, 3))  # wider to accommodate two images side-by-side
    sag_slice = 6* matrix_c[0] // 10
    img_tmp = crop_center(np.moveaxis(img_sk_lr_no_moco, 0, -1),(matrix_c[0], matrix_c[1], int(res_scale*matrix_e[2])))
    img_insp = np.rot90(normalize(abs(img_tmp[..., nphase//2])), k=1)  
    img_exp = np.rot90(normalize(abs(img_tmp[..., 0])), k=1)
    sag_exp = np.rot90(img_exp[sag_slice, :, :], k=2)
    sag_insp = np.rot90(img_insp[sag_slice, :, :], k=2)
    concat_img = np.concatenate((sag_exp, sag_insp), axis=1)
    im = ax.imshow(concat_img, cmap='gray', vmax=1)
    ax.axis('off')
    fig.savefig(folder + f'/sketching/png/sketched_img_{nch}_coils_lr_no_moco.png', bbox_inches='tight', pad_inches=0)
    plt.show()
        
    # Start monitoring
    print("MONITORING GPU USAGE...")
    tic, toc, memory_usage_non_sketched, shutdown = monitor_gpu_memory(gpu_index=devnum)
    tic()
        
    img_lr_no_moco = mvc(scale*sk4d.SketchedLowRankRecon4D(y=ksp_us/scale, 
                                                mps=mps, 
                                                lamda=lamda, 
                                                reduced_ncoils=nc,
                                    weights=dens_us, 
                                    coord=coord_us,
                                    moco=False,
                                    # solver='GradientMethod', # Default
                                    solver='PrimalDualHybridGradient', # Default
                                    toeplitz=toeplitz,
                                    show_pbar=True, device=sp.Device(devnum),
                                    number_non_sketched_coils=nc-1, # None = Defaults to nch-1 
                                    max_init_iter=max_init_iter + (max_outer_iter * (max_inner_iter-1)),
                                    max_outer_iter=0,
                                    max_inner_iter=0,
                                    max_power_iter=30,
                                    save_objective_values=False, # This is memory intensive as it requires saving y on the backend, and computing A^Hy at every iteration
                                    coil_batch_size=nc,
                                    ).run())

    ni_img = nib.Nifti1Image(crop_center(abs(np.moveaxis(img_lr_no_moco, 0, -1)), matrix_c), affine=aff)
    nib.save(ni_img, folder + '/sketching/img_lr_no_moco')
    
    toc()
    # Plot results
    plt.figure(figsize=(3,4))
    plt.plot(memory_usage_non_sketched)
    plt.xlabel('time')
    plt.ylabel('memory usage (MB)')
    plt.grid(True)
    plt.show()
    shutdown()
    print("GPU MONITORING COMPLETE.")

    # Clear variables stored on GPU 
    import cupy as cp
    cp._default_memory_pool.free_all_blocks()

    import sigpy.plot as pl
    vmax = np.percentile(abs(img_lr_no_moco.ravel()), 95)
    for resp in range(nphase):
        pl.ImagePlot(img_lr_no_moco[resp,...], 
                    x=1, y=3,
                    title="Under-sampled Reconstruction (LR - no registration)", 
                    colormap="gray", 
                    vmax=vmax,
                    )

# Try a 4D Motion Compensated Low Rank (MoCo-LR) reconstruction

Note, for this reconstruction, we did not make use of G inside the iterative solver. It is possible to set G to the motion-fields registration operator (i.e. an operator that registers all images together when applied, and does the inverse when adjointed). However, this is a risky operation as the adjoint-forward combination likely does not recover the original input. For few iterations, this may be okay. However, for larger number of iterations, a smarter workaround may be needed.

We proposed a solution that applies the motion-compensation inside the low rank proximal operator. Critically, this approach only uses forward transformations, by iterating through respiratory states and registering all other states to the current state, applying low rank soft thresholding, and then only extracting the current state. It is a little slower, but with a GPU, it's not too bad...

In [3]:
if mocolor:
    import sketching_4d_app as sk4d
    import importlib
    importlib.reload(sk4d)
    importlib.reload(sp)
    import sketching_app
    importlib.reload(sketching_app)
    import prox_ltr
    importlib.reload(prox_ltr)
    import registration_oflow3D
    importlib.reload(registration_oflow3D)
    import opticalflow3D
    importlib.reload(opticalflow3D)
    
    # Get memory usage statistics
    mempool = cp.get_default_memory_pool()
    pinned_mempool = cp.get_default_pinned_memory_pool()
    mempool.free_all_blocks()
    pinned_mempool.free_all_blocks()
    cp._default_memory_pool.free_all_blocks()
    cp.cuda.stream.get_current_stream().synchronize() 
    
    # Start monitoring
    print("MONITORING GPU USAGE...")
    tic, toc, memory_usage_sketched, shutdown = monitor_gpu_memory(gpu_index=devnum)
    tic()

    np.random.seed(1)
    img_sk_lr_moco_jacobian = mvc(scale*sk4d.SketchedMotionCompensatedLowRankRecon4D(y=ksp_us/scale, 
                                                mps=mps, 
                                                lamda=lamda, 
                                                reduced_ncoils=nch,
                                    weights=dens_us, 
                                    coord=coord_us,
                                    moco=True,
                                    N_frames_per_block=nphase,
                                    # solver='GradientMethod', # Default
                                    solver='PrimalDualHybridGradient', # Default
                                    toeplitz=toeplitz,
                                    show_pbar=True, device=sp.Device(devnum),
                                    number_non_sketched_coils=number_non_sketched_coils, # None = Defaults to nch-1 
                                    max_init_iter=max_init_iter,
                                    max_outer_iter=max_outer_iter,
                                    max_inner_iter=max_inner_iter,
                                    max_power_iter=30,
                                    save_objective_values=False, # This is memory intensive as it requires saving y on the backend, and computing A^Hy at every iteration
                                    coil_batch_size=nch,
                                    ).run())

    ni_img = nib.Nifti1Image(crop_center(abs(np.moveaxis(img_sk_lr_moco_jacobian, 0, -1)), matrix_c), affine=aff)
    nib.save(ni_img, folder + '/sketching/img_sk_lr_moco_jacobian')
    
    sketched_memory_lor_moco = np.array(memory_usage_sketched)*1e-3
    toc()
    # Plot results
    plt.figure(figsize=(3,4))
    plt.plot(memory_usage_sketched)
    plt.xlabel('time')
    plt.ylabel('memory usage (MB)')
    plt.grid(True)
    plt.show()
    shutdown()
    print("GPU MONITORING COMPLETE.")

    # Clear variables stored on GPU 
    import cupy as cp
    cp._default_memory_pool.free_all_blocks()

    import sigpy.plot as pl
    vmax = np.percentile(abs(img_sk_lr_moco_jacobian.ravel()), 95)
    for resp in range(nphase):
        pl.ImagePlot(img_sk_lr_moco_jacobian[resp,...], 
                    x=1, y=3,
                    title="Under-sampled Sketching Reconstruction (LR - moco registration)", 
                    colormap="gray", 
                    vmax=vmax,
                    )
        
        
    # Display sagittal slices from insp/exp phases
    print(f"Sketched reconstruction with {nch} coils:")
    fig, ax = plt.subplots(1, 1, figsize=(3, 3))  # wider to accommodate two images side-by-side
    sag_slice = 6* matrix_c[0] // 10
    img_tmp = crop_center(np.moveaxis(img_sk_lr_moco_jacobian, 0, -1),(matrix_c[0], matrix_c[1], int(res_scale*matrix_e[2])))
    img_insp = np.rot90(normalize(abs(img_tmp[..., nphase//2])), k=1)  
    img_exp = np.rot90(normalize(abs(img_tmp[..., 0])), k=1)
    sag_exp = np.rot90(img_exp[sag_slice, :, :], k=2)
    sag_insp = np.rot90(img_insp[sag_slice, :, :], k=2)
    concat_img = np.concatenate((sag_exp, sag_insp), axis=1)
    im = ax.imshow(concat_img, cmap='gray', vmax=1)
    ax.axis('off')
    fig.savefig(folder + f'/sketching/png/sketched_img_{nch}_coils_mocolr.png', bbox_inches='tight', pad_inches=0)
    plt.show()
        

NameError: name 'mocolor' is not defined

## Let's try a 4D Locally Low Rank recon (without registration)

If interested, we also implemented a locally low rank reconstruction. We did not use it in the paper. But it could be of some use if interested!

In [ ]:
if llor:
    import sketching_4d_app as sk4d
    import importlib
    importlib.reload(sk4d)
    importlib.reload(sp)

    np.random.seed(1)
    img_sk_llr_no_moco = mvc(scale*sk4d.SketchedLocallyLowRankRecon4D(y=ksp_us/scale, 
                                                mps=mps, 
                                                lamda=lamda, 
                                                reduced_ncoils=nch,
                                    weights=dens_us, 
                                    coord=coord_us,
                                    solver='GradientMethod', # Default
                                    toeplitz=toeplitz,
                                    show_pbar=True, device=sp.Device(devnum),
                                    number_non_sketched_coils=number_non_sketched_coils, # None = Defaults to nch-1 
                                    max_init_iter=max_init_iter,
                                    max_outer_iter=max_outer_iter,
                                    max_inner_iter=max_inner_iter,
                                    max_power_iter=30,
                                    save_objective_values=False, # This is memory intensive as it requires saving y on the backend, and computing A^Hy at every iteration
                                    coil_batch_size=nch,
                                    ).run())
    
    ni_img = nib.Nifti1Image(crop_center(abs(np.moveaxis(img_sk_llr_no_moco, 0, -1)), matrix_c), affine=aff)
    nib.save(ni_img, folder + '/sketching/img_sk_llr_no_moco')

    # Clear variables stored on GPU 
    import cupy as cp
    cp._default_memory_pool.free_all_blocks()

    import sigpy.plot as pl
    vmax = np.percentile(abs(img_sk_llr_no_moco.ravel()), 95)
    pl.ImagePlot(img_sk_llr_no_moco[0,...], 
                x=1, y=3,
                title="Under-sampled Sketching Reconstruction (LLR - no registration)", 
                colormap="gray", 
                vmax=vmax,
                )
    
    # Display sagittal slices from insp/exp phases
    print(f"Sketched reconstruction with {nch} coils:")
    fig, ax = plt.subplots(1, 1, figsize=(3, 3))  # wider to accommodate two images side-by-side
    sag_slice = 6* matrix_c[0] // 10
    img_tmp = crop_center(np.moveaxis(img_sk_llr_no_moco, 0, -1),(matrix_c[0], matrix_c[1], int(res_scale*matrix_e[2])))
    img_insp = np.rot90(normalize(abs(img_tmp[..., nphase//2])), k=1)  
    img_exp = np.rot90(normalize(abs(img_tmp[..., 0])), k=1)
    sag_exp = np.rot90(img_exp[sag_slice, :, :], k=2)
    sag_insp = np.rot90(img_insp[sag_slice, :, :], k=2)
    concat_img = np.concatenate((sag_exp, sag_insp), axis=1)
    im = ax.imshow(concat_img, cmap='gray', vmax=1)
    ax.axis('off')
    fig.savefig(folder + f'/sketching/png/sketched_img_{nch}_coils_llr_no_moco.png', bbox_inches='tight', pad_inches=0)
    plt.show()
    
    img_llr_no_moco = mvc(scale*sk4d.SketchedLocallyLowRankRecon4D(y=ksp_us/scale, 
                                                mps=mps, 
                                                lamda=lamda, 
                                                reduced_ncoils=nc,
                                    weights=dens_us, 
                                    coord=coord_us,
                                    solver='GradientMethod', # Default
                                    toeplitz=toeplitz,
                                    show_pbar=True, device=sp.Device(devnum),
                                    number_non_sketched_coils=nc-1, # None = Defaults to nch-1 
                                    max_init_iter=max_init_iter + (max_outer_iter * (max_inner_iter-1)),
                                    max_outer_iter=0,
                                    max_inner_iter=0,
                                    max_power_iter=30,
                                    save_objective_values=False, # This is memory intensive as it requires saving y on the backend, and computing A^Hy at every iteration
                                    coil_batch_size=nc,
                                    ).run())

    ni_img = nib.Nifti1Image(crop_center(abs(np.moveaxis(img_llr_no_moco, 0, -1)), matrix_c), affine=aff)
    nib.save(ni_img, folder + '/sketching/img_llr_no_moco')

    # Clear variables stored on GPU 
    import cupy as cp
    cp._default_memory_pool.free_all_blocks()

    import sigpy.plot as pl
    vmax = np.percentile(abs(img_llr_no_moco.ravel()), 95)
    pl.ImagePlot(img_llr_no_moco[0,...], 
                x=1, y=3,
                title="Under-sampled Reconstruction (LLR - no registration)", 
                colormap="gray", 
                vmax=vmax,
                )

# Let's plot some images


In [ ]:

# Define helper functions
def normalize(img, percentile=95):
    """Normalize the image to a given percentile."""
    img_n = img / np.percentile(img.ravel(), percentile)
    return img_n

def crop_2d(image, matrix_size):
    """Crop the center of a 2D image to a given size."""
    center = [dim // 2 for dim in image.shape]
    half_size = [msize // 2 for msize in matrix_size]
    slices = tuple(slice(c - h, c + h) for c, h in zip(center, half_size))
    return image[slices]

# Plot images
for resp in range(nphase):
    image = np.rot90(abs(img_sk_cg_sense[resp,...]), 2)
    tmp_slice = int(50*res_scale)  # Coronal slice (x, y)
    sag_slice = int(128*res_scale)   # Sagittal slice (y, z)
    voxel_crop = int(60*res_scale) 
    side_crop = -20

    # Extract slices for coronal and sagittal views
    coronal_view = normalize(image[voxel_crop:-voxel_crop, voxel_crop:-voxel_crop,tmp_slice])  # Coronal (x, y)
    sagittal_view = normalize(image[voxel_crop:-voxel_crop, sag_slice, :side_crop])  # Sagittal (y, z)

    # Concatenate views
    concatenated_view = np.concatenate((coronal_view, sagittal_view), axis=1)

    # Plotting
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(concatenated_view, cmap='gray', vmax=1)
    ax.axis('off')
    # ax.set_title("Coronal and Sagittal Views", fontsize=16, fontweight='bold')

    # Save the plot
    output_folder = folder + "/sketching/png/"
    os.makedirs(output_folder, exist_ok=True)
    output_path = os.path.join(output_folder, f"CG-SENSE_concatenated_phase_{resp}.png")
    fig.savefig(output_path, bbox_inches='tight', pad_inches=0)
    plt.show()

    # Plotting
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(coronal_view, cmap='gray', vmax=1)
    ax.axis('off')
    # ax.set_title("Coronal and Sagittal Views", fontsize=16, fontweight='bold')

    # Save the plot
    output_folder = folder + "/sketching/png/"
    os.makedirs(output_folder, exist_ok=True)
    output_path = os.path.join(output_folder, f"CG-SENSE_coronal_phase_{resp}.png")
    fig.savefig(output_path, bbox_inches='tight', pad_inches=0)
    plt.show()

    print(f"Saved concatenated views to {output_path}")
    
    
    # Next image...
    image = np.rot90(abs(img_sk_tv[resp,...]), 2)

    # Extract slices for coronal and sagittal views
    coronal_view = normalize(image[voxel_crop:-voxel_crop, voxel_crop:-voxel_crop,tmp_slice])  # Coronal (x, y)
    sagittal_view = normalize(image[voxel_crop:-voxel_crop, sag_slice, :side_crop])  # Sagittal (y, z)

    # Concatenate views
    concatenated_view = np.concatenate((coronal_view, sagittal_view), axis=1)

    # Plotting
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(concatenated_view, cmap='gray', vmax=1)
    ax.axis('off')
    # ax.set_title("Coronal and Sagittal Views", fontsize=16, fontweight='bold')

    # Save the plot
    output_folder = folder + "/sketching/png/"
    os.makedirs(output_folder, exist_ok=True)
    output_path = os.path.join(output_folder, f"TV_concatenated_phase_{resp}.png")
    fig.savefig(output_path, bbox_inches='tight', pad_inches=0)
    plt.show()

    # Plotting
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(coronal_view, cmap='gray', vmax=1)
    ax.axis('off')
    # ax.set_title("Coronal and Sagittal Views", fontsize=16, fontweight='bold')

    # Save the plot
    output_folder = folder + "/sketching/png/"
    os.makedirs(output_folder, exist_ok=True)
    output_path = os.path.join(output_folder, f"TV_coronal_phase_{resp}.png")
    fig.savefig(output_path, bbox_inches='tight', pad_inches=0)
    plt.show()

    print(f"Saved concatenated views to {output_path}")
    
    
    # Next image...
    image = np.rot90(abs(img_sk_lr_no_moco[resp,...]), 2)

    # Extract slices for coronal and sagittal views
    coronal_view = normalize(image[voxel_crop:-voxel_crop, voxel_crop:-voxel_crop,tmp_slice])  # Coronal (x, y)
    sagittal_view = normalize(image[voxel_crop:-voxel_crop, sag_slice, :side_crop])  # Sagittal (y, z)

    # Concatenate views
    concatenated_view = np.concatenate((coronal_view, sagittal_view), axis=1)

    # Plotting
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(concatenated_view, cmap='gray', vmax=1)
    ax.axis('off')
    # ax.set_title("Coronal and Sagittal Views", fontsize=16, fontweight='bold')

    # Save the plot
    output_folder = folder + "/sketching/png/"
    os.makedirs(output_folder, exist_ok=True)
    output_path = os.path.join(output_folder, f"LOR_concatenated_phase_{resp}.png")
    fig.savefig(output_path, bbox_inches='tight', pad_inches=0)
    plt.show()

    # Plotting
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(coronal_view, cmap='gray', vmax=1)
    ax.axis('off')
    # ax.set_title("Coronal and Sagittal Views", fontsize=16, fontweight='bold')

    # Save the plot
    output_folder = folder + "/sketching/png/"
    os.makedirs(output_folder, exist_ok=True)
    output_path = os.path.join(output_folder, f"LOR_coronal_phase_{resp}.png")
    fig.savefig(output_path, bbox_inches='tight', pad_inches=0)
    plt.show()

    print(f"Saved concatenated views to {output_path}")
    
    
    # Next image...
    image = np.rot90(abs(img_sk_lr_moco_jacobian[resp,...]), 2)

    # Extract slices for coronal and sagittal views
    coronal_view = normalize(image[voxel_crop:-voxel_crop, voxel_crop:-voxel_crop,tmp_slice])  # Coronal (x, y)
    sagittal_view = normalize(image[voxel_crop:-voxel_crop, sag_slice, :side_crop])  # Sagittal (y, z)

    # Concatenate views
    concatenated_view = np.concatenate((coronal_view, sagittal_view), axis=1)

    # Plotting
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(concatenated_view, cmap='gray', vmax=1)
    ax.axis('off')
    # ax.set_title("Coronal and Sagittal Views", fontsize=16, fontweight='bold')

    # Save the plot
    output_folder = folder + "/sketching/png/"
    os.makedirs(output_folder, exist_ok=True)
    output_path = os.path.join(output_folder, f"MOCOLOR_concatenated_phase_{resp}.png")
    fig.savefig(output_path, bbox_inches='tight', pad_inches=0)
    plt.show()

    # Plotting
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(coronal_view, cmap='gray', vmax=1)
    ax.axis('off')
    # ax.set_title("Coronal and Sagittal Views", fontsize=16, fontweight='bold')

    # Save the plot
    output_folder = folder + "/sketching/png/"
    os.makedirs(output_folder, exist_ok=True)
    output_path = os.path.join(output_folder, f"MOCOLOR_coronal_phase_{resp}.png")
    fig.savefig(output_path, bbox_inches='tight', pad_inches=0)
    plt.show()

    print(f"Saved concatenated views to {output_path}")